# Genetic Algorithm for Function Optimization
This notebook implements a Genetic Algorithm (GA) to find the maximum value of a mathematical function.

## Run the Genetic Algorithm

In [6]:
import numpy as np

# 1. Define the Objective Function to Optimize
def objective_function(x):
    return x * np.sin(10 * np.pi * x) + 1.0

# 2. Initialize Parameters
POP_SIZE = 200          # Population size
GENOMES_LEN = 32        # Length of binary chromosome
CROSSOVER_RATE = 0.8    # Probability of crossover
MUTATION_RATE = 0.05    # Probability of mutating a gene
GENERATIONS = 100       # Number of generations
BOUNDS = [-1, 2]        # [min, max] of x
TOURNAMENT_SIZE = 3     # Tournament size for selection

# Decode binary chromosome back to a real number in the bounds
def decode_chromosome(chromosome, bounds, bits_len):
    binary_str = ''.join(map(str, chromosome))
    decimal_val = int(binary_str, 2)
    precision = (bounds[1] - bounds[0]) / (2**bits_len - 1)
    return bounds[0] + decimal_val * precision

# 3. Create Initial Population
def initialize_population(pop_size, bits_len):
    return np.random.randint(2, size=(pop_size, bits_len))

# 4. Evaluate Fitness
def evaluate_fitness(population, bounds, bits_len):
    decoded = np.array([decode_chromosome(ind, bounds, bits_len) for ind in population])
    fitness = objective_function(decoded)
    min_fit = np.min(fitness)
    if min_fit < 0:
        fitness = fitness - min_fit + 1e-6
    return fitness, decoded

# 5. Selection (Tournament Selection)
def tournament_selection(population, fitness, tournament_size):
    selected = []
    pop_size = len(population)
    for _ in range(pop_size):
        candidates = np.random.choice(pop_size, size=tournament_size, replace=False)
        best_candidate_idx = candidates[np.argmax(fitness[candidates])]
        selected.append(population[best_candidate_idx])
    return np.array(selected)

# 6. Crossover (Single-point crossover)
def crossover(parent1, parent2, crossover_rate):
    if np.random.rand() < crossover_rate:
        point = np.random.randint(1, len(parent1))
        child1 = np.concatenate((parent1[:point], parent2[point:]))
        child2 = np.concatenate((parent2[:point], parent1[point:]))
        return child1, child2
    return parent1.copy(), parent2.copy()

# 7. Mutation
def mutate(chromosome, mutation_rate):
    for i in range(len(chromosome)):
        if np.random.rand() < mutation_rate:
            chromosome[i] = 1 - chromosome[i]
    return chromosome

# 8. Iteration (Evolution Loop)
population = initialize_population(POP_SIZE, GENOMES_LEN)
best_fitness_history = []
best_x_history = []

for generation in range(GENERATIONS):
    fitness, decoded = evaluate_fitness(population, BOUNDS, GENOMES_LEN)

    # Track the best solution in this generation
    best_idx = np.argmax(fitness)
    best_x = decoded[best_idx]
    best_val = objective_function(best_x)
    best_fitness_history.append(best_val)
    best_x_history.append(best_x)

    # Selection
    selected_population = tournament_selection(population, fitness, TOURNAMENT_SIZE)

    # Reproduction (Crossover and Mutation)
    next_generation = []
    for i in range(0, POP_SIZE, 2):
        p1, p2 = selected_population[i], selected_population[min(i+1, POP_SIZE-1)]
        c1, c2 = crossover(p1, p2, CROSSOVER_RATE)
        next_generation.append(mutate(c1, MUTATION_RATE))
        next_generation.append(mutate(c2, MUTATION_RATE))

    population = np.array(next_generation)[:POP_SIZE]

# 9. Output the Best Solution
best_overall_idx = np.argmax(best_fitness_history)
best_overall_generation = best_overall_idx + 1
best_overall_x = best_x_history[best_overall_idx]
best_overall_val = best_fitness_history[best_overall_idx]

print("==================================================")
print("          GENETIC ALGORITHM OPTIMIZATION          ")
print("==================================================")
print(f"Best Solution Found in Generation: {best_overall_generation} / {GENERATIONS}")
print(f"Optimal parameter x              : {best_overall_x:.8f}")
print(f"Maximum function value f(x)      : {best_overall_val:.8f}")
print("==================================================")

          GENETIC ALGORITHM OPTIMIZATION          
Best Solution Found in Generation: 72 / 100
Optimal parameter x              : 1.85054747
Maximum function value f(x)      : 2.85027377
